# Retrieve Supabase Templates

Experiment to retrieve rows from the `templates` table in Supabase.

In [3]:
from dotenv import load_dotenv
from supabase import create_client
import os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_KEY must be defined in the .env file.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Connected to Supabase")

Connected to Supabase


In [2]:
response = supabase.table("templates").select("*").execute()
templates = response.data
print(f"Retrieved {len(templates)} templates.")

Retrieved 5 templates.


In [3]:
if templates:
    import json
    print(json.dumps(templates[0], indent=2))

{
  "id": "zettelkasten",
  "name": "Zettelkasten",
  "description": "Atomic note format emphasizing one idea per note and explicit links between concepts.",
  "instructions": "Capture a single idea per note. Use concise language and create links to related concepts whenever possible.",
  "category": "Knowledge Management",
  "tags": [
    "notes",
    "knowledge-management",
    "atomic-notes",
    "research"
  ],
  "version": 1,
  "preview_markdown": "---\nid: 202606210201\ntitle: \"Quantum Entanglement\"\ntags: [physics, quantum-mechanics]\naliases: [Spooky Action]\n---\n\n# Quantum Entanglement\n\nQuantum entanglement describes the phenomenon where two particles become correlated in such a way that measuring one affects the state of the other.\n\n## Related Concepts\n\n- [[Wave Function Collapse]]\n- [[Quantum Computing]]\n",
  "template_markdown": "---\nid: [NOTE_ID]\ntitle: \"[TITLE]\"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise description of a single c

In [4]:
from IPython.display import display, Markdown

# Render a simple Markdown string
display(Markdown(templates[0]['template_markdown']))


---
id: [NOTE_ID]
title: "[TITLE]"
tags: [tag1, tag2]
aliases: []
---

# [TITLE]

Write a concise description of a single concept.

## Related Concepts

- [[Related Note 1]]
- [[Related Note 2]]


In [5]:
from IPython.display import display, Markdown

# Render a simple Markdown string
display(Markdown(templates[1]['preview_markdown']))


# Topic: TCP Handshake

## 1. Plain Explanation

> Explain it as if speaking to a child. Avoid jargon.

Before two computers can talk, they first greet each other and agree that both are ready to communicate.

## 2. Identified Knowledge Gaps

- [ ] Unsure why the ACK packet is required.
- [ ] Need to review retransmissions.

## 3. Refined Source Material

- [RFC 793](https://datatracker.ietf.org/doc/html/rfc793) — Defines TCP behavior.


--- 
---


# Learner & Parser Agent Drafts

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agents.note_profiler.app.agent import root_agent
from agents.note_profiler.app.models import TemplateProfile, NoteProfile
from google.adk.apps import App
from google.adk.runners import InMemoryRunner
from google.genai import types


In [6]:
from agents.note_profiler.app.agent import create_agent

# Use OpenAI via LiteLLM
# openai_agent = create_agent(model_provider="openai")

# Use built-in Gemini
gemini_agent = create_agent(model_provider="gemini")


In [10]:

app = App(name="app", root_agent=gemini_agent)
runner = InMemoryRunner(app=app)

session = await runner.session_service.create_session(
    app_name="app", user_id="dev_user"
)
refactored_templates = []


# Draft pipeline execution to Note Profiler Agent:
for note in templates[:]:
    agent_input = f"Name: {note.get('name', '')}\nInstructions: {note.get('instructions', '')}\nTemplate: {note.get('template_markdown', '')}\nExample: {note.get('preview_markdown', '')}"
    
    print(f"Running Note Profiler for {note.get('name')}...\n")
    enriched_profile = None
    
    async for event in runner.run_async(
        user_id="dev_user",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part.from_text(text=agent_input)]),
    ):
        # 1. Check for the structured Pydantic output
        if event.actions.state_delta.get('profile_data') is not None:
            enriched_profile = session.state['profile_data'] = event.actions.state_delta['profile_data']
            
            note.update(enriched_profile)

        # 2. Print the raw text/errors from the LLM to diagnose why output is None
        if event.content and event.content.parts:
            print(f"RAW EVENT CONTENT: {event.content.parts[0].text[:500]}...")
            
    print("\n" + "="*40)
    print(f"Final State profile_data: {session.state.get('profile_data') is not None}")
    print(f"Final Enriched Profile: {type(enriched_profile)}")
    print("="*40 + "\n")
    
    # if enriched_profile:
    #     print(enriched_profile.model_dump_json(indent=2))
    # else:
    #     print("No structured output was generated. Please review the RAW EVENT CONTENT above.")

refactored_templates.append(enriched_profile)

Running Note Profiler for Zettelkasten...

RAW EVENT CONTENT: {
  "name": "Zettelkasten",
  "description": "This template is designed for implementing the Zettelkasten method, focusing on capturing single, atomic ideas. Its primary purpose is permanent knowledge capture, deep learning, and building an interconnected web of concepts. It is best suited for individuals or teams who want to build a robust, evergreen knowledge base, track individual ideas, and explore relationships between diverse topics. The template contains a unique identifier, a clear title...

Final State profile_data: True
Final Enriched Profile: <class 'dict'>

Running Note Profiler for Feynman Technique...

RAW EVENT CONTENT: {
  "name": "Feynman Technique",
  "description": "This template embodies the Feynman Technique, a powerful method for achieving deep understanding and effective learning of any topic. It guides users through the process of explaining a concept in simple terms, actively identifying areas of con

In [15]:
list(templates[0].keys())

['id',
 'name',
 'description',
 'instructions',
 'category',
 'tags',
 'version',
 'preview_markdown',
 'template_markdown',
 'purpose',
 'sections',
 'organization_structure',
 'style',
 'embedding_text',
 'embedding_vector']

In [13]:
list(refactored_templates[0].keys())

['name',
 'description',
 'instructions',
 'tags',
 'purpose',
 'sections',
 'organization_structure',
 'style']

---

### 

### SAVE AS initial_data.jsonl
(before updating db)

In [17]:

import ast
from pathlib import Path

# Resolve project root directory
PROJECT_ROOT = Path().resolve().parents[1]
PROJECT_ROOT


WindowsPath('C:/Users/Home/Desktop/projetos/mark-me-down')

### Get embedding text 

Concat all keys in yaml style

In [21]:
# Path to the data file
import jsonlines

jsonl_path = PROJECT_ROOT / "dev" / "data" / "templates-enriched-data.jsonl"


# Open the file and write all dictionaries at once
with jsonlines.open(jsonl_path, mode="w") as writer:
    writer.write_all(templates)


In [25]:
# Path to the data file
# Read and parse the templates from the python-literal-structured jsonl file
with jsonlines.open(jsonl_path, "r") as reader:
    refactored_templates = list(reader)
refactored_templates
    

[{'id': 'zettelkasten',
  'name': 'Zettelkasten',
  'description': "This template is designed for implementing the Zettelkasten method, focusing on capturing single, atomic ideas. Its primary purpose is permanent knowledge capture, deep learning, and building an interconnected web of concepts. It is best suited for individuals or teams who want to build a robust, evergreen knowledge base, track individual ideas, and explore relationships between diverse topics. The template contains a unique identifier, a clear title, descriptive tags for classification, alternative names (aliases), and a concise explanation of a single concept. A dedicated section allows for explicit linking to related concepts, fostering a networked thinking approach. Information is organized with a clear, hierarchical structure, using explicit sections and bullet lists for connections. The style is concise, factual, and objective, emphasizing clarity and easy retrieval of information. It's ideal for creating highly 

In [27]:
list(refactored_templates[0].keys())

['id',
 'name',
 'description',
 'instructions',
 'category',
 'tags',
 'version',
 'preview_markdown',
 'template_markdown',
 'purpose',
 'sections',
 'organization_structure',
 'style',
 'embedding_text',
 'embedding_vector']

### Build Semantic doc (embedding_text)
yaml stylem

In [ ]:
from textwrap import dedent

def note_profile_to_embedding_text(profile: NoteProfile) -> str:
    """Serialize a NoteProfile into a deterministic text for embeddings."""

    def yaml_list(items: list[str]) -> str:
        return "\n".join(f"  - {item}" for item in items)

    return dedent(f"""\
    purpose:
    {yaml_list(profile.purpose)}

    contains:
    {yaml_list(profile.contains)}

    organization:
    {yaml_list(profile.organization)}

    style:
    {yaml_list(profile.style)}
    """).strip()

In [76]:
from agents.note_profiler.app.models import TemplateProfile, NoteProfile


In [ ]:
list(refactored_templates[0].keys())

['id',
 'name',
 'description',
 'instructions',
 'category',
 'tags',
 'version',
 'preview_markdown',
 'template_markdown',
 'purpose',
 'sections',
 'organization_structure',
 'style',
 'embedding_text',
 'embedding_vector']

In [13]:
# Building Template Profile semantic doc 
from textwrap import dedent

def note_profile_to_embedding_text(profile: dict) -> str:
    """Serialize a NoteProfile into a deterministic text for embeddings."""

    def yaml_list(items: list[str]) -> str:
        return "\n".join(f"  - {item}" for item in items)

    parts = []
    
    # Map of output topic header to the profile attribute name
    topics = [
        ('name','name'),
        ('description','description'),
        ('instructions','instructions'),
        ('category','category'),
        ('tags','tags'),
        ('version','version'),
        # ('preview_markdown','preview_markdown'), #Example
        # ('template_markdown','template_markdown'), # Template Model
        ('purpose','purpose'),
        ('sections','sections'),
        ('organization_structure','organization_structure'),
        ('style','style'),
        ( 'raw_note_text','raw_note_text')
    ]
    
    for topic_name, attr_name in topics:
        if topic_name in profile:
            val = profile[attr_name]
            # Ensure the value exists, is not an empty string, and (if list) is not empty
            if val is not None and val != "" and (not isinstance(val, list) or len(val) > 0):
                formatted_list = yaml_list(val) if isinstance(val, list) else f"  - {val}"
                parts.append(f"{topic_name}:\n{formatted_list}")
                
    return "\n\n".join(parts).strip()


In [33]:
for note in refactored_templates:

    note['embedding_text'] = note_profile_to_embedding_text(note) 

# Print or inspect the first result
print(note['embedding_text'])


name:
  - Keep a Changelog

description:
  - This template is designed to meticulously document and track all modifications made to a project, product, or system across different versions. It offers a structured and chronological record of updates, making it an indispensable tool for understanding the evolution of a codebase or product. It's best suited for generating formal release notes, maintaining a transparent history of product changes, or providing detailed documentation of software updates. The template captures essential information such as version numbers, release dates, and systematically categorizes each change into distinct types like new features, improvements, bug fixes, removals, deprecations, and security patches. The information is organized hierarchically by release, with detailed bulleted lists under each change category, facilitating easy navigation and comprehension of what has been altered in each iteration. The style is objective, technical, and concise, ensurin

### Vectorize
(before updating db)

In [37]:
docs = [note['embedding_text'] for note in refactored_templates]
docs

["name:\n  - Zettelkasten\n\ndescription:\n  - This template is designed for implementing the Zettelkasten method, focusing on capturing single, atomic ideas. Its primary purpose is permanent knowledge capture, deep learning, and building an interconnected web of concepts. It is best suited for individuals or teams who want to build a robust, evergreen knowledge base, track individual ideas, and explore relationships between diverse topics. The template contains a unique identifier, a clear title, descriptive tags for classification, alternative names (aliases), and a concise explanation of a single concept. A dedicated section allows for explicit linking to related concepts, fostering a networked thinking approach. Information is organized with a clear, hierarchical structure, using explicit sections and bullet lists for connections. The style is concise, factual, and objective, emphasizing clarity and easy retrieval of information. It's ideal for creating highly interlinked, atomic n

In [45]:
len([i.values for  i in response.embeddings])

5

In [46]:
for n,note in enumerate(refactored_templates):

    note['embedding_vector'] = response.embeddings[n].values




### Final SAVE
(updating db)

In [49]:
jsonl_path

WindowsPath('C:/Users/Home/Desktop/projetos/mark-me-down/dev/data/templates-enriched-data.jsonl')

In [ ]:
all OK  -Updateing Existing Template Models at db  

- save model results as new inital data 
+ include other keys 

- re create TemplateModelProfile
***
TemplateProfile is used : 1) to update templates table 2 ) later to match NoteProfile
NoteProfile used to RAG (search , compare and retrieve)
(so, no previews nor models are used only semantic search
***


- DO NOT add other keys like preview markdown to embeded text  

- get embedding_text locally by concating all keys into yaml format

- vecotrize embedding text 

- drop fields @table templates: difficulty

- drop fields @templates model: preview,  difficulty


--- 
---


# Search Similarity and Retrival Engine


In [6]:
note_profile = NoteProfile(
    name="",
    description= "",
    # # (
    #     "A rough technical planning note describing implementation ideas for a "
    #     "software project. The note mixes requirements, architectural decisions, "
    #     "implementation tasks and future ideas. Information is fragmented and "
    #     "would benefit from organization into logical sections while preserving "
    #     "all technical details."
    # ),
    instructions="",
    raw_note_text="""
Need login pagee,Supabase auth,Google login maybe? (check poros andcons)
+email verification
pricing page
check RLS docs
migration after MVP (this will be a bitch, need to furher break down)
""",
    tags=[
        "technical",
        "software",
        "project",
        "planning",
        "implementation",
        "architecture",
        "requirements",
    ],
    purpose=[
        "plan implementation",
        "capture technical ideas",
        "track future work",
    ],
    sections=[
        "requirements",
        "authentication",
        "architecture",
        "implementation tasks",
        "future work",
    ],
    organization_structure=[
        "hierarchical sections",
        "group related concepts",
        "task lists",
        "logical ordering",
    ],
    style=[
        "technical",
        "concise",
        "fragmented",
        "markdown",
    ],
)

In [11]:
note_profile.model_dump()

{'name': '',
 'description': '',
 'instructions': '',
 'raw_note_text': '\nNeed login pagee,Supabase auth,Google login maybe? (check poros andcons)\n+email verification\npricing page\ncheck RLS docs\nmigration after MVP (this will be a bitch, need to furher break down)\n',
 'tags': ['technical',
  'software',
  'project',
  'planning',
  'implementation',
  'architecture',
  'requirements'],
 'purpose': ['plan implementation',
  'capture technical ideas',
  'track future work'],
 'sections': ['requirements',
  'authentication',
  'architecture',
  'implementation tasks',
  'future work'],
 'organization_structure': ['hierarchical sections',
  'group related concepts',
  'task lists',
  'logical ordering'],
 'style': ['technical', 'concise', 'fragmented', 'markdown']}

In [ ]:
query_text = note_profile_to_embedding_text(note_profile.model_dump()) 


In [15]:
# Markdown(query_text)
print(query_text)

tags:
  - technical
  - software
  - project
  - planning
  - implementation
  - architecture
  - requirements

purpose:
  - plan implementation
  - capture technical ideas
  - track future work

sections:
  - requirements
  - authentication
  - architecture
  - implementation tasks
  - future work

organization_structure:
  - hierarchical sections
  - group related concepts
  - task lists
  - logical ordering

style:
  - technical
  - concise
  - fragmented
  - markdown

raw_note_text:
  - 
Need login pagee,Supabase auth,Google login maybe? (check poros andcons)
+email verification
pricing page
check RLS docs
migration after MVP (this will be a bitch, need to furher break down)


In [ ]:
query_vector =  

In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=query_text,
    config=types.EmbedContentConfig(
        output_dimensionality=1536
    ),
)


In [17]:

# note_profile.embedding_vector = response.embeddings[0].values
embedding_vector = response.embeddings[0].values


In [18]:
from supabase import create_client

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY,
)

results = (
    supabase
    .rpc(
        "match_templates",
        {
            "query_embedding": embedding_vector,   # list[float]
            "match_count": 5,
        },
    )
    .execute()
)

for template in results.data:
    print(template["name"], template["similarity"])

Architectural Decision Record 0.693203213522834
Zettelkasten 0.678610202224789
Keep a Changelog 0.6727800248987
Feynman Technique 0.665380246728926
SOAP Note 0.660417605174867


In [23]:
# setattr(note_profile,'embedding_vector', embedding_vector)
results.data

[{'id': 'adr',
  'name': 'Architectural Decision Record',
  'description': "This template is designed to capture and document significant technical and architectural decisions, their underlying context, the rationale for the chosen approach, and their anticipated consequences. It serves as a historical record for critical design choices, enabling teams to understand 'why' certain paths were taken and to track their long-term impact. It is best for documenting software architecture changes, resolving design dilemmas, ensuring team alignment on technical direction, and maintaining a clear log of key architectural evolution. The template contains information regarding the problem statement, influencing factors, the specific decision made, and both the positive and negative implications. Its structure logically progresses from problem definition to solution and impact assessment, often using bulleted lists for detailed consequences. The style is typically professional, objective, and analy

--- 
---


# Selector Agent Drafts

In [ ]:
- adapt profiler prompt

In [ ]:
---

['id',
 'name',
 'description',
 'instructions',
 'category',
 'tags',
 'version',
 'preview_markdown',
 'template_markdown',
 'purpose',
 'sections',
 'organization_structure',
 'style',
 'embedding_text',
 'embedding_vector']